# Phase 8 B v3 — replay identitas yang direview

Notebook ini membaca pengukuran detector/tracker/ReID yang sudah tersimpan. Tidak ada inference model baru. Review pengguna dan AI berbukti pixel diterapkan hanya pada video, ID, keluarga pengukuran, dan interval frame yang dicatat. B v2 tetap utuh.

ID tampilan P/M/C/T + empat digit adalah ID **passage**, bukan ID tracker atau identitas fisik. Pedestrian hanya menerima role positif; `person` yang belum diketahui perannya tidak otomatis pedestrian. Rider menambah 0. Kendaraan tetap memakai BC bbox kendaraan mentah.


In [ ]:
from pathlib import Path
import json, sys, unittest

# Set ke checkout GitHub yang memuat revisi ini. Jangan memasukkan token ke notebook.
REPO_ROOT = Path("/kaggle/working/traffic-vehicle-counter-IDM-LOC")
SOURCE_RUN_ROOT = None  # Auto-discover exact immutable donor manifest below /kaggle/input.
SOURCE_VIDEO_ROOT = None  # Auto-discover actual TGDI video files.
INPUT_ROOT = "/kaggle/input"
VIDEO_IDS = None  # None = all eight. For a first pilot: ["weekday_pagi"].
RENDER_ANNOTATED_VIDEO = True
OUTPUT_ROOT = "/kaggle/working/results/phase_8_b_v3"

sys.path.insert(0, str(REPO_ROOT / "notebooks"))
assert (REPO_ROOT / "results/b_v3/input/review_decisions.json").is_file()


## Validasi software sebelum replay

Tes sintetis memeriksa invariant dan sumber pengukuran; bukan benchmark akurasi. Kegagalan menghentikan replay.


In [ ]:
modules = ["phase_8_b_v2_counting_tests", "phase_8_b_v2_annotation_tests", "phase_8_b_v2_batch_tests",
           "phase_8_b_v3_counting_tests", "phase_8_b_v3_annotation_tests", "phase_8_b_v3_batch_tests"]
suite = unittest.TestSuite(unittest.defaultTestLoader.loadTestsFromName(name) for name in modules)
validation = unittest.TextTestRunner(verbosity=1).run(suite)
assert validation.wasSuccessful() and not validation.skipped, "Software validation failed or skipped"


## Replay sumber terverifikasi

Lampirkan dataset `chrisbiran/tgdi-traffic-videos` dan output notebook `chrisbiran/phase-8-tgdi-full-video-research` sebagai input Kaggle. Notebook donor yang tepat ditentukan SHA manifest, tidak memilih run terbaru secara spekulatif. CPU cukup; GPU tidak diperlukan untuk replay counting dan render.

Semua parameter geometri, margin 25 px, dan timer 60 frame dipertahankan. Satu unit fisik dapat mempunyai beberapa ID mentah hanya dengan relasi terverifikasi sesuai scope. Scope review merupakan keputusan/assumsi yang eksplisit; bukan ground truth formal.


In [ ]:
from phase_8_b_v3_batch import run_b_v3_batch

report = run_b_v3_batch(repo_root=REPO_ROOT, source_run_root=SOURCE_RUN_ROOT,
    source_video_root=SOURCE_VIDEO_ROOT, input_root=INPUT_ROOT, video_ids=VIDEO_IDS,
    output_root=OUTPUT_ROOT, render=RENDER_ANNOTATED_VIDEO)
print(json.dumps({key: report.get(key) for key in ("status", "output_directory", "frames_processed",
    "operational_total", "pending_unique_unit_episode_passages", "unknown_person_role_keys",
    "new_detector_tracker_reid_execution")}, indent=2))


## Berkas untuk review

Di output run baru terdapat `batch_report.json`, snapshot sembilan executable, salinan keputusan review, dan retention manifest. Setiap video mempunyai `counting_result.json`, `pending_groups.json`, `person_role_review_queue.json`, `comparison_b_v2.json`, serta video anotasi/laporan render jika aktif.

Review kategori kasus, bukan seluruh baris satu per satu. Periksa juga accepted timeout: penurunan pending atau kenaikan jumlah operational belum membuktikan reliabilitas. `comparison_b_v2.json` hanya membandingkan ledger dengan penyebut pending yang dijelaskan; tidak menyatakan perbaikan akurasi.

Hasil bersifat offline setelah episode/video selesai. Tidak ada tuning detector, tracker, ReID, stabilisasi, atau deployment dalam notebook ini.
